In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [3]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [4]:
files

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

In [5]:
shared_path = "../Data/R3Data"

### 3. patient_data_ie
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [6]:
idx = 3
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

patient_data_ie
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'ORG_NAME', 'EXAM_CODE', 'EXAM_NAME', 'COMPLETED_DATE', 'ORDER_DATE', 'EXAM_TYPE', 'FIRST_MAMMO_IND', 'LAST_ACTIVITY_DATE', 'INTERNAL_EXAM_ID', 'PATIENT_ID']


In [7]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['patient_data_ie.txt', 'patient_data_ie.csv', 'patient_data_ie.csv']

### Cancer

In [8]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [9]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

/var/folders/g9/k5r102q54f126nrlr692zy6c0000gn/T/ipykernel_4260/483784814.py:10: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  current_df = pd.read_csv(file_path, encoding='latin-1')


✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [10]:
final_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_CODE,ORG_NAME,EXAM_CODE,EXAM_NAME,EXAM_MODIFIER,EXAM_STATUS,SCHEDULED_DATE,COMPLETED_DATE,FINALIZED_DATE,BEGIN_DATE,ORDER_DATE,EXAM_TYPE_ID,EXAM_SIDE,EXAM_REQUESTOR,EXAM_TYPE,SELF_REQUEST_IND,EQUIPMENT_CLEAN_IND,FIRST_MAMMO_IND,LAST_SCREEN_MONTH_CD,LAST_CBE_CD,LAST_CBE_DATE,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4334469937,69423837,MWHOAK,Magee Womens Imaging Oakland,MRBRSTBIWX,MR BREAST BILATERAL WWO CONTRAST WWO CAD,NOREMINDER/EPIC,F,06/25/2020,06/25/2020,06/25/2020,06/25/2020,06/17/2020,NaN,B,"MCAULIFFE, PRISCILLA",Breast MRI,N,NaN,N,NaN,NONE,NaN,02/12/2021,83749463,1152783
1,4334469937,69511096,MWHMNV,Magee Womens Imaging Monroeville,BUBRSTLMRT,BREAST ULTRASOUND LIMITED RIGHT,NaN,F,06/09/2020,06/09/2020,06/11/2020,06/09/2020,NaN,NaN,R,"THOMAS, ROBERT D, MD",Ultrasound,N,NaN,N,NaN,NONE,NaN,02/12/2021,83836589,1152783
2,4334469937,69525464,MWHMNV,Magee Womens Imaging Monroeville,BIDIADBITO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,EPIC,F,06/09/2020,06/09/2020,06/11/2020,06/09/2020,06/08/2020,NaN,B,"DOYLE, GRETCHEN E",Mammography,N,Y,N,NaN,NONE,NaN,02/12/2021,83841700,1152783
3,4334469937,69510810,MWHHIL,Magee Womens Imaging Hillman,BUBXBRNCCR,US BX BREAST NEEDLE CORE W CLIP PLACE RT WWO S...,NaN,F,06/11/2020,06/11/2020,06/16/2020,06/11/2020,NaN,NaN,R,"SHAHEEN, ANNE",Interventional procedure,N,Y,N,NaN,NONE,NaN,02/12/2021,83825254,1152783
4,4334469937,69599771,MWHHIL,Magee Womens Imaging Hillman,BIDIADGPPR,DIAG MAMMO PROD DIRECT DIGITAL IMAGES RT POST ...,NaN,F,06/11/2020,06/11/2020,06/16/2020,06/11/2020,NaN,NaN,R,"SHAHEEN, ANNE",Digital Mammography,N,Y,N,NaN,NONE,NaN,02/12/2021,83804449,1152783
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120283,4335413505,450177891,MWHSTH,Magee Womens Imaging South,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/09/2022,12/09/2022,12/11/2022,12/09/2022,09/28/2022,NaN,B,"MASTASCUSA, BETH",Mammography,N,Y,N,NaN,NONE,NaN,02/10/2023,73347889,616683
120284,4333651277,459402348,MWHSTH,Magee Womens Imaging South,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/30/2022,12/30/2022,01/01/2023,12/30/2022,11/23/2022,NaN,B,"TEPE, PAUL G, MD",Mammography,N,Y,N,NaN,NONE,NaN,12/30/2022,73780040,9034157
120285,4333260965,451934907,HMTHIC,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/28/2022,12/28/2022,12/30/2022,12/28/2022,07/13/2022,NaN,B,"KEELER, JORDAN LEAH",Mammography,N,NaN,N,NaN,NONE,NaN,05/08/2023,74229878,6155024
120286,4333119414,450097209,MWHMNV,Magee Womens Imaging Monroeville,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/27/2022,12/27/2022,12/28/2022,12/27/2022,10/04/2022,NaN,B,"MILLBAUER, LINDA L",Mammography,N,NaN,N,NaN,NONE,NaN,01/22/2023,73368129,2186707


In [11]:
final_df['COMPLETED_DATE'] = pd.to_datetime(final_df['COMPLETED_DATE'], format='%m/%d/%Y')
final_df['COMPLETED_DATE'] = final_df['COMPLETED_DATE'].dt.strftime('%Y-%m-%d')
final_df['ORDER_DATE'] = pd.to_datetime(final_df['ORDER_DATE'], format='%m/%d/%Y')
final_df['ORDER_DATE'] = final_df['ORDER_DATE'].dt.strftime('%Y-%m-%d')

In [20]:
final_df['LAST_ACTIVITY_DATE'] = pd.to_datetime(final_df['LAST_ACTIVITY_DATE'], format='%m/%d/%Y')
final_df['LAST_ACTIVITY_DATE'] = final_df['LAST_ACTIVITY_DATE'].dt.strftime('%Y-%m-%d')

In [12]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [21]:
unique_df = final_df.copy()

In [22]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [23]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [24]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [25]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_CODE,ORG_NAME,EXAM_CODE,EXAM_NAME,EXAM_MODIFIER,EXAM_STATUS,SCHEDULED_DATE,COMPLETED_DATE,FINALIZED_DATE,BEGIN_DATE,ORDER_DATE,EXAM_TYPE_ID,EXAM_SIDE,EXAM_REQUESTOR,EXAM_TYPE,SELF_REQUEST_IND,EQUIPMENT_CLEAN_IND,FIRST_MAMMO_IND,LAST_SCREEN_MONTH_CD,LAST_CBE_CD,LAST_CBE_DATE,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID,duplicate_count
0,4330018595,63027507,MWHNTH,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,07/26/2019,2019-07-26,07/28/2019,07/26/2019,2019-07-08,NaN,B,"RAHNAMAY-AZAR, KRISTINA M",Mammography,N,Y,Y,NaN,NONE,NaN,2021-01-05,90728533,9654707,1
1,4330018595,63737104,MWHNTH,Magee Womens Imaging North,BIDIADBITO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,EPIC,F,08/19/2019,2019-08-19,08/19/2019,08/19/2019,2019-08-05,NaN,B,"RAHNAMAY-AZAR, KRISTINA M",Mammography,N,Y,N,NaN,NONE,NaN,2021-01-05,99424137,9654707,1
2,4330018595,60103700,MWHNTH,Magee Womens Imaging North,BIDIADRTTO,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,EPIC,F,06/02/2020,2020-06-02,06/02/2020,06/02/2020,2020-03-26,NaN,R,"RAHNAMAY-AZAR, KRISTINA M",Mammography,N,Y,N,NaN,NONE,NaN,2021-01-05,83422682,9654707,1
3,4330018595,60695725,MWHNTH,Magee Womens Imaging North,BUBRSTLMRT,BREAST ULTRASOUND LIMITED RIGHT,NaN,F,06/02/2020,2020-06-02,06/02/2020,06/02/2020,NaN,NaN,R,"RAHNAMAY-AZAR, KRISTINA M",Ultrasound,N,Y,N,NaN,NONE,NaN,2021-01-05,83910173,9654707,1
4,4330018595,60690108,MWHNTH,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,LIMITEDEX,F,06/02/2020,2020-06-02,06/02/2020,06/02/2020,NaN,NaN,B,"RAHNAMAY-AZAR, KRISTINA M",Mammography,N,Y,N,NaN,NONE,NaN,2021-01-05,83905509,9654707,1
5,4330018595,60667802,MWHNTH,Magee Womens Imaging North,BIDIADGPPR,DIAG MAMMO PROD DIRECT DIGITAL IMAGES RT POST ...,NaN,F,06/05/2020,2020-06-05,06/10/2020,06/05/2020,NaN,NaN,R,"RAHNAMAY-AZAR, KRISTINA M",Digital Mammography,N,Y,N,NaN,NONE,NaN,2021-01-05,83973670,9654707,1
6,4330018595,60693047,MWHNTH,Magee Womens Imaging North,BUBXBRNCCR,US BX BREAST NEEDLE CORE W CLIP PLACE RT WWO S...,NaN,F,06/05/2020,2020-06-05,06/10/2020,06/05/2020,NaN,NaN,R,"RAHNAMAY-AZAR, KRISTINA M",Interventional procedure,N,Y,N,NaN,NONE,NaN,2021-01-05,83918365,9654707,1
7,4330018595,69461777,MWHOAK,Magee Womens Imaging Oakland,MRBRSTBIWX,MR BREAST BILATERAL WWO CONTRAST WWO CAD,EPIC,F,06/30/2020,2020-06-30,07/01/2020,06/30/2020,2020-06-23,NaN,B,"DIEGO, EMILIA JOSEFA",Breast MRI,N,NaN,N,NaN,NONE,NaN,2021-01-05,83657913,9654707,1
8,4330018595,69089910,MWHOAK,Magee Womens Imaging Oakland,BURSLRTEA,ULTRASOUND GUIDED RADIOACTIVE SEED LOC RIGHT E...,NaN,F,07/28/2020,2020-07-28,07/28/2020,07/28/2020,NaN,NaN,R,"DIEGO, EMILIA JOSEFA",Interventional procedure,N,Y,N,NaN,NONE,NaN,2021-01-05,82371743,9654707,1
9,4330018595,69089931,MWHOAK,Magee Womens Imaging Oakland,BURSLRT,ULTRASOUND GUIDED RADIOACTIVE SEED LOCALIZATIO...,NaN,F,07/28/2020,2020-07-28,07/28/2020,07/28/2020,NaN,NaN,R,"DIEGO, EMILIA JOSEFA",Interventional procedure,N,Y,N,NaN,NONE,NaN,2021-01-05,82371867,9654707,1


In [26]:
unique_df['duplicate_count'].unique()

array([1])

In [27]:
unique_df[extract_cols]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_NAME,EXAM_CODE,EXAM_NAME,COMPLETED_DATE,ORDER_DATE,EXAM_TYPE,FIRST_MAMMO_IND,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4330018595,63027507,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2019-07-26,2019-07-08,Mammography,Y,2021-01-05,90728533,9654707
1,4330018595,63737104,Magee Womens Imaging North,BIDIADBITO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,2019-08-19,2019-08-05,Mammography,N,2021-01-05,99424137,9654707
2,4330018595,60103700,Magee Womens Imaging North,BIDIADRTTO,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,2020-06-02,2020-03-26,Mammography,N,2021-01-05,83422682,9654707
3,4330018595,60695725,Magee Womens Imaging North,BUBRSTLMRT,BREAST ULTRASOUND LIMITED RIGHT,2020-06-02,NaN,Ultrasound,N,2021-01-05,83910173,9654707
4,4330018595,60690108,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2020-06-02,NaN,Mammography,N,2021-01-05,83905509,9654707
...,...,...,...,...,...,...,...,...,...,...,...,...
120283,4339959661,68434457,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2020-11-09,2020-09-14,Mammography,N,2020-12-22,82737072,7269813
120284,4339959661,454148687,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2021-11-10,2021-09-30,Mammography,N,2021-11-22,87437824,7269813
120285,4339959661,454612462,UPMC Hamot Breast Imaging,BIDIADLTTO,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,2021-11-15,2021-11-11,Mammography,N,2021-11-22,87910122,7269813
120286,4339959661,453550821,UPMC Hamot Breast Imaging,BIBXBVACLT,BREAST STEREO BX VAC ASSIST LT WWO SPEC IMG,2021-11-22,NaN,Digital Mammography,N,2021-11-22,87859992,7269813


In [28]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)

### Control

In [29]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['patient_data_ie_controls.txt', 'patient_data_ie_controls.csv']

In [30]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [31]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

/var/folders/g9/k5r102q54f126nrlr692zy6c0000gn/T/ipykernel_4260/483784814.py:10: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  current_df = pd.read_csv(file_path, encoding='latin-1')


✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [32]:
final_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_CODE,ORG_NAME,EXAM_CODE,EXAM_NAME,EXAM_MODIFIER,EXAM_STATUS,SCHEDULED_DATE,COMPLETED_DATE,FINALIZED_DATE,BEGIN_DATE,ORDER_DATE,EXAM_TYPE_ID,EXAM_SIDE,EXAM_REQUESTOR,EXAM_TYPE,SELF_REQUEST_IND,EQUIPMENT_CLEAN_IND,FIRST_MAMMO_IND,LAST_SCREEN_MONTH_CD,LAST_CBE_CD,LAST_CBE_DATE,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID
0,4334585594,68008794,HMTHIC,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/08/2020,12/08/2020,12/09/2020,12/08/2020,10/21/2020,NaN,B,"NAGURNEY, EMILY M",Mammography,N,NaN,N,NaN,NONE,NaN,02/25/2021,81307232,45497467
1,4334453170,61626590,MWHOAK,Magee Womens Imaging Oakland,BUBRSTLMRT,BREAST ULTRASOUND LIMITED RIGHT,EPIC,F,07/11/2020,07/11/2020,07/11/2020,07/11/2020,02/07/2020,NaN,R,"SCRUGGS, KATHERINE L",Ultrasound,N,NaN,N,NaN,NONE,NaN,03/02/2021,84088623,2505470
2,4334453170,61762111,MWHOAK,Magee Womens Imaging Oakland,BUBRSTLMRT,BREAST ULTRASOUND LIMITED RIGHT,NaN,F,02/04/2020,02/04/2020,02/04/2020,02/04/2020,NaN,NaN,R,"SCRUGGS, KATHERINE L",Ultrasound,N,Y,N,NaN,NONE,NaN,03/02/2021,84022507,2505470
3,4334453170,61305273,MWHOAK,Magee Womens Imaging Oakland,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,01/18/2020,01/18/2020,01/25/2020,01/18/2020,12/06/2019,NaN,B,"VOLKAR, JUDITH K, MD",Mammography,N,Y,N,NaN,NONE,NaN,03/02/2021,85746212,2505470
4,4334653987,60088335,HMTHIC,UPMC Hamot Imaging Center,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,07/07/2020,07/07/2020,07/23/2020,07/07/2020,04/16/2020,NaN,B,"GREEN, STACY J",Mammography,N,NaN,N,NaN,NONE,NaN,07/07/2020,83309087,45514480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293092,4334306940,450218288,MWHMNV,Magee Womens Imaging Monroeville,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/01/2022,12/01/2022,12/05/2022,12/01/2022,09/15/2022,NaN,B,"MILLBAUER, LINDA L",Mammography,N,Y,N,NaN,NONE,NaN,06/09/2023,73596134,9439497
293093,4330109074,459138767,MWHMNV,Magee Womens Imaging Monroeville,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/20/2022,12/20/2022,12/21/2022,12/20/2022,12/19/2022,NaN,B,"WEAVER, SIMA A",Mammography,N,Y,N,NaN,NONE,NaN,06/02/2023,72418532,3017117
293094,4333020539,450787967,MWHMNV,Magee Womens Imaging Monroeville,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/30/2022,12/30/2022,01/02/2023,12/30/2022,10/31/2022,NaN,B,"YURKO, CHRISTINA R",Mammography,N,Y,N,NaN,NONE,NaN,04/21/2023,73951489,9719418
293095,4333076287,450143208,MWHMNV,Magee Womens Imaging Monroeville,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,EPIC,F,12/09/2022,12/09/2022,12/15/2022,12/09/2022,09/21/2022,NaN,B,"HYNES, MARY ANN",Mammography,N,Y,N,NaN,NONE,NaN,06/06/2023,73422035,3044268


In [33]:
final_df['COMPLETED_DATE'] = pd.to_datetime(final_df['COMPLETED_DATE'], format='%m/%d/%Y')
final_df['COMPLETED_DATE'] = final_df['COMPLETED_DATE'].dt.strftime('%Y-%m-%d')
final_df['ORDER_DATE'] = pd.to_datetime(final_df['ORDER_DATE'], format='%m/%d/%Y')
final_df['ORDER_DATE'] = final_df['ORDER_DATE'].dt.strftime('%Y-%m-%d')
final_df['LAST_ACTIVITY_DATE'] = pd.to_datetime(final_df['LAST_ACTIVITY_DATE'], format='%m/%d/%Y')
final_df['LAST_ACTIVITY_DATE'] = final_df['LAST_ACTIVITY_DATE'].dt.strftime('%Y-%m-%d')

In [35]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [36]:
unique_df = final_df.copy()

In [37]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [38]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [39]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [40]:
unique_df['duplicate_count'].unique()

array([1])

In [41]:
unique_df[unique_extract_cols]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,ORG_NAME,EXAM_CODE,EXAM_NAME,COMPLETED_DATE,ORDER_DATE,EXAM_TYPE,FIRST_MAMMO_IND,LAST_ACTIVITY_DATE,INTERNAL_EXAM_ID,PATIENT_ID,duplicate_count
0,4330000534,78048053,Magee Womens Imaging South,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,2018-03-28,NaN,Mammography,N,2021-03-09,6909744,8267555,1
1,4330000534,64787403,Magee Womens Imaging North,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2019-05-18,2019-05-18,Mammography,N,2021-03-09,90405175,8267555,1
2,4330000534,63280481,Magee Womens Imaging South,BUBRSTCMRT,BREAST ULTRASOUND COMPLETE RIGHT,2019-06-20,NaN,Ultrasound,N,2021-03-09,90988303,8267555,1
3,4330000534,63280358,Magee Womens Imaging South,BUBRSTCMLT,BREAST ULTRASOUND COMPLETE LEFT,2019-06-20,NaN,Ultrasound,N,2021-03-09,90988393,8267555,1
4,4330000534,63433254,Magee Womens Imaging South,BIFU,BREAST IMAGING MISCELLANEOUS FOLLOW UP TECHNIC...,2019-06-20,2019-06-20,Digital Mammography,N,2021-03-09,90276137,8267555,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
293092,4339982047,450915296,Magee Womens Imaging South,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2022-10-12,2022-10-11,Mammography,N,2023-06-14,73297957,7794108,1
293093,4339988199,65403287,UPMC Hamot Fairview Office,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2018-11-28,NaN,Mammography,N,2021-02-25,93436075,45307048,1
293094,4339988199,61697072,UPMC Hamot West Ridge Office,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2020-02-21,2020-02-10,Mammography,N,2021-02-25,84949771,45307048,1
293095,4339988199,67718778,UPMC Hamot West Ridge Office,BISCRDBITO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,2021-02-25,2021-02-19,Mammography,N,2021-02-25,80006891,45307048,1


In [42]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)